# Create Card Strength Input data

**Workflow:**
- Load and filter raw Scryfall data for Commander-legal cards.
- Normalize and clean card text for transformer inputs.
- Generate text embeddings using pre-trained sentence transformers.
- Engineer numerical and boolean features from card attributes.
- Export the final dataset for model training.

In [ ]:
import pandas as pd
import numpy as np
import re
import ast
from typing import Dict, Any
from sentence_transformers import SentenceTransformer

from common.dataset_filter import filter_dataset
from common.text_normalizing import normalize_symbol

In [ ]:
INPUT_DATA_PATH = "../data/scryfall/oracle-cards-20251022090300.json"
OUTPUT_DATA_PATH = "../data/training/card_strength_input.parquet"

In [ ]:
df = pd.read_json(INPUT_DATA_PATH)
df = filter_dataset(df)

print(f"shape: {df.shape}")
df.head()

## Handle Multi-face Cards

- Define `prepare_transformer_input` to flatten multi-face cards into a single text string.
- Clean reminder text and normalize card name references to 'this card'.
- Apply the transformation to create a `transformer_input` column.

In [ ]:
def prepare_oracle_text_for_transformer(text: str, card_name: str, remove_reminder_text: bool = False) -> str:
    if not isinstance(text, str):
        return ""

    # TODO: investigate if reminder text removal yields better results
    # remove reminder text
    if remove_reminder_text:
        text = re.sub(pattern=r"\([^)]*\)", repl="", string=text)

    # replace newlines
    text = text.replace(" \n", ". ")
    text = text.replace("\n", ". ")

    # normalize bullet points
    text = text.replace("•", " option: ")

    # normalize choice separators
    text = re.sub(pattern=r"choose\s+(one|two|one or both)\s+—", repl=r"choose \1 :", string=text, flags=re.IGNORECASE)

    # replace symbols
    text = re.sub(pattern=r"\{([^}]*)\}", repl=normalize_symbol, string=text)

    # fix spacing
    text = re.sub(pattern=r"\s+", repl=" ", string=text)
    text = re.sub(pattern=r"\.\s*\.", repl=".", string=text)
    text = text.replace(" ,", ",").replace(" .", ".").replace(" :", ":")
    text = text.lower().strip()

    # replace card name
    text = text.replace(card_name.lower(), "this card")
    return text

In [ ]:
def prepare_transformer_input(row: pd.Series) -> str:
    """ Creates a structured, natural language input for the transformer. """
    
    # get release year from row
    release_year = "unknown year"
    if 'released_at' in row and pd.notna(row['released_at']):
        release_year = str(row['released_at'])[:4]

    def get_face_text(face_data: Dict[str, Any], card_name: str, rarity: str, release_year: str, keywords: list) -> str:
        name = face_data.get('name', card_name).lower()

        # normalize mana cost symbols to natural language
        raw_cost = face_data.get('mana_cost', '')
        if raw_cost == "":
            cost_str = "no mana cost"
        else:
            cost_str = re.sub(pattern=r"\{([^}]*)\}", repl=normalize_symbol, string=raw_cost)
            cost_str = re.sub(pattern=r"\s+", repl=" ", string=cost_str).strip()
            
        cmc = int(face_data.get('cmc', 0))
        type_line = face_data.get('type_line', 'unknown type').lower()
        
        raw_oracle = face_data.get('oracle_text', '')
        clean_oracle = prepare_oracle_text_for_transformer(text=raw_oracle, card_name=name, remove_reminder_text=False)
        
        # add stats
        stats_str = ""
        if 'power' in face_data and pd.notna(face_data['power']):
            stats_str = f" STATS: {face_data['power']} power and {face_data['toughness']} toughness."
        elif 'loyalty' in face_data and pd.notna(face_data['loyalty']):
            stats_str = f" LOYALTY: {face_data['loyalty']}."
            
        # add keywords section
        keywords_str = ""
        if keywords and len(keywords) > 0:
            keywords_str = f" KEYWORDS: {', '.join(keywords).lower()}."

        return f"CARD: {name}. COST: {cost_str} ({cmc} mana). TYPE: {type_line}. RARITY: {rarity}. RELEASED: {release_year}.{stats_str}{keywords_str} TEXT: {clean_oracle}"

    rarity = str(row.get('rarity', 'unknown')).lower()
    keywords = row.get('keywords', [])
    
    # handle multi-face cards
    if 'card_faces' in row and isinstance(row['card_faces'], list):
        face_texts = [get_face_text(face_data=face, card_name=row['name'], rarity=rarity, release_year=release_year, keywords=keywords) for face in row['card_faces']]
        return " // ".join(face_texts)
    
    # handle normal cards
    return get_face_text(face_data=row.to_dict(), card_name=row['name'], rarity=rarity, release_year=release_year, keywords=keywords)

In [ ]:
df['transformer_input'] = df.apply(prepare_transformer_input, axis=1)
print("\n".join(df.sample(20)['transformer_input'].to_list()))

## Generate Text Embeddings

In [ ]:
def create_embeddings(df: pd.DataFrame) -> pd.DataFrame:
    model = SentenceTransformer(model_name_or_path='Alibaba-NLP/gte-modernbert-base', trust_remote_code=True)

    input_prompts = df['transformer_input'].to_list()
    print(f"Embedding {len(input_prompts)} cards...")
    embeddings = model.encode(sentences=input_prompts, batch_size=32, show_progress_bar=True)

    embed_cols = [f"embed_{i}" for i in range(embeddings.shape[1])]
    embeddings_df = pd.DataFrame(data=embeddings, columns=embed_cols, index=df.index)
    
    df_result = pd.concat(objs=[df, embeddings_df], axis=1)
    return df_result

In [ ]:
df = create_embeddings(df=df)
df.head()

## Tabular Feature Engineering

- **Numerical:** Extract CMC, Power, Toughness, Loyalty, and Color count.
- **Boolean:** Flag specific card types (Planeswalker, Legendary, Saga) and attributes (X-cost, Modal, Transform).

In [ ]:
def safe_to_float(val):
    # TODO: nan is not allowed for MLP
    if pd.isna(val):
        return np.nan
    val = str(val).replace('*', '0').replace('+', '')
    try:
        return float(val)
    except ValueError:
        return np.nan
    
def count_colors(val):
    if isinstance(val, list):
        return len(val)
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                return len(parsed)
        except (ValueError, SyntaxError):
            pass
    return 0

def check_is_modal(row):
    if '•' in str(row.get('oracle_text', '')):
        return 1
    faces = row.get('card_faces')
    if isinstance(faces, list) and len(faces) > 1:
        all_spells = all(('Instant' in str(face.get('type_line', '')) or 'Sorcery' in str(face.get('type_line', ''))) for face in faces)
        if all_spells:
            return 1
    return 0

def check_is_transform(row):
    faces = row.get('card_faces')
    if isinstance(faces, list) and len(faces) > 1:
        all_permanents = all(('Instant' not in str(face.get('type_line', '')) and 'Sorcery' not in str(face.get('type_line', ''))) for face in faces)
        if all_permanents:
            return 1
    return 0

def map_rarity(rarity):
    rarity_map = {'common': 1, 'uncommon': 2, 'rare': 3, 'mythic': 4}
    return rarity_map.get(str(rarity).lower(), 0)

def get_word_count(text):
    if not isinstance(text, str):
        return 0
    return len(re.findall(r'\w+', text))

df['num_cmc'] = df['cmc'].apply(safe_to_float)
df['num_power'] = df['power'].apply(safe_to_float)
df['num_toughness'] = df['toughness'].apply(safe_to_float)
df['num_loyalty'] = df['loyalty'].apply(safe_to_float)
df['num_colors'] = df['color_identity'].apply(count_colors)
df['num_rarity'] = df['rarity'].apply(map_rarity)
df['num_release_year'] = pd.to_datetime(df['released_at']).dt.year
df['num_word_count'] = df['oracle_text'].apply(get_word_count)

df['bool_is_w'] = df['color_identity'].apply(lambda x: 1 if 'W' in (x if isinstance(x, list) else []) else 0)
df['bool_is_u'] = df['color_identity'].apply(lambda x: 1 if 'U' in (x if isinstance(x, list) else []) else 0)
df['bool_is_b'] = df['color_identity'].apply(lambda x: 1 if 'B' in (x if isinstance(x, list) else []) else 0)
df['bool_is_r'] = df['color_identity'].apply(lambda x: 1 if 'R' in (x if isinstance(x, list) else []) else 0)
df['bool_is_g'] = df['color_identity'].apply(lambda x: 1 if 'G' in (x if isinstance(x, list) else []) else 0)

df['bool_is_game_changer'] = df['game_changer'].fillna(False).astype(int)
df['bool_power_var'] = df['power'].astype(str).str.contains(r'[\*X]', na=False).astype(int)
df['bool_toughness_var'] = df['toughness'].astype(str).str.contains(r'[\*X]', na=False).astype(int)
bool_cols = ['Planeswalker', 'Legendary', 'Saga', 'Class']
for col in bool_cols:
    df[f'bool_is_{col.lower()}'] = df['type_line'].astype(str).str.contains(col, case=False, na=False).astype(int)

df['bool_has_x_cost'] = df['mana_cost'].astype(str).str.contains('{X}', case=False, na=False).astype(int)
df['bool_is_colorless'] = df['color_identity'].apply(lambda x: 1 if (isinstance(x, list) and len(x) == 0) or str(x) == '[]' else 0)
df['bool_is_modal'] = df.apply(check_is_modal, axis=1)
df['bool_is_transform'] = df.apply(check_is_transform, axis=1)

boolean_cols = [col for col in df.columns if col.startswith('bool_')]
numerical_cols = [col for col in df.columns if col.startswith('num_')]

df.head()

## Dataset Export

- Concatenate all engineered features into a single final DataFrame.
- Save the resulting dataset as a Parquet file for high-performance retrieval during training.

In [ ]:
print(f"Final shape {df.shape}")
print(f"Exporting to {OUTPUT_DATA_PATH}...")
df.to_parquet(OUTPUT_DATA_PATH, index=False)